# 13 - When does accDM become indistinguishable from CDM?

The accelerated-DM daughter is born with a velocity kick `v = sqrt(eta(eta+2))/(1+eta)`
set by the energy boost `eta_acc`. Adopting the physical relation **`eta = 1e11 / m`**
(`m` = daughter mass in GeV), a heavier daughter has smaller `eta`, free-streams less,
and is colder. This notebook sweeps `m` upward and finds the mass above which accDM's
matter power `P(k)` and lensed CMB spectra match "CDM" within our thresholds.

Two references ("CDM"): (1) the **cold limit** of the same model (`eta -> 0`), isolating
free-streaming; (2) **plain LCDM** (no acc species). Two metric families: (A) a fixed
fractional tolerance on the spectra, and (B) a cosmic-variance-limited chi^2 detectability.

Daughter on exact quadrature (fluid closure is unusable). Hybrid notebook: inline unit
asserts for the pure metrics (nbmake) + a CLASS scan + diagnostic plots. Run in the
`accDM` classy environment: `pytest --nbmake notebooks_test/13_test_accDM_CDM_indistinguishability.ipynb`.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from classy import Class

plt.rcParams.update({
    'mathtext.fontset': 'stix', 'font.family': 'serif', 'font.size': 11,
    'axes.labelsize': 12, 'legend.fontsize': 9, 'lines.linewidth': 1.5, 'figure.dpi': 300})
qual_colors = ['#377eb8', '#ff7f00', '#4daf4a', '#f781bf', '#984ea3']

# ---- Planck 2018 base cosmology ----------------------------------
omega_b, omega_cdm0 = 0.022383, 0.12011
A_s, n_s, tau_reio, H0 = 2.1005829616811546e-9, 0.96605, 0.0543, 67.32
base_params = {'omega_b': omega_b, 'omega_cdm': omega_cdm0, 'H0': H0,
               'A_s': A_s, 'n_s': n_s, 'tau_reio': tau_reio}

# ---- fixed accDM decay sector ------------------------------------
KAPPA, A_T = 2.0, 0.13
A_REC = 1.0 / (1.0 + 1090.0)
ETA_COLD = 1e-12                     # "cold limit" eta

# ---- scan axes ---------------------------------------------------
F_SEQ = [0.01, 0.05, 0.1]
MASS_GRID = np.logspace(11, 19, 12)  # GeV  -> eta ~ 1 down to ~1e-8

# ---- observables -------------------------------------------------
K_NODES = np.logspace(-3, 0.0, 60)   # 1/Mpc, sub-horizon at z=0
L_MAX = 2500
Q_BINS = 250                          # daughter momentum bins (accuracy vs speed knob)

PREC_COMMON = {'evolver': 0, 'reionization_z_start_max': 80, 'background_Nloga': 2001}
PREC_PK  = {**PREC_COMMON, 'output': 'mPk', 'P_k_max_1/Mpc': 1.0, 'z_max_pk': 0.0}
PREC_CMB = {**PREC_COMMON, 'output': 'tCl,pCl,lCl,mPk', 'lensing': 'yes',
            'l_max_scalars': L_MAX, 'P_k_max_1/Mpc': 1.0, 'z_max_pk': 0.0}

# ---- survey / detectability assumptions --------------------------
V_SURVEY = 3.0e3**3                    # ~(3 Gpc)^3, Euclid-like fiducial; edit to taste
F_SKY = 0.7

def eta_of(m):
    return 1e11 / m


In [ ]:
def ocdm_rescaled(f_acc):
    """CDM density rescaled for the decayed daughter, as in notebook 5."""
    return omega_cdm0 * (1 + f_acc*(1 - A_REC**KAPPA)/(1 + (A_REC/A_T)**KAPPA))**(-1)

def lcdm_params():
    """Plain LCDM, no acc species; a massive-nu sector matched to the accDM runs."""
    p = dict(base_params); p.update(PREC_CMB)
    p.update({'N_ncdm': 1, 'deg_ncdm': 3, 'm_ncdm': 0.02, 'T_ncdm': 0.71611,
              'ncdm_quadrature_strategy': 0, 'ncdm_N_momentum_bins': 15, 'N_ur': 0.00441})
    return p

def _accdm_common(f_acc, m, eta):
    p = dict(base_params); p.update(PREC_CMB)
    p.update({'omega_cdm': ocdm_rescaled(f_acc),
              'vary_Gamma_acc': 'yes', 'kappa_acc': KAPPA, 'a_t_acc': A_T,
              'f_acc': f_acc, 'eta_acc': eta,
              'm_acc_in_GeV': m, 'm_cdm_in_GeV': m,
              'N_ncdm': 2, 'deg_ncdm': '3, 1',
              'm_ncdm': '0.02, {:.6e}'.format(m*1e9),
              'T_ncdm': '0.71611, 1', 'ncdm_quadrature_strategy': '0, 4',
              'ncdm_N_momentum_bins': '15, {:d}'.format(Q_BINS), 'N_ur': 0.00441,
              'ncdm_fluid_approximation': 0,
              'ncdm_fluid_trigger_rho_accDM_over_rho_dcdm': 10})
    return p

def accdm_params(f_acc, m):
    return _accdm_common(f_acc, m, eta_of(m))

def coldlimit_params(f_acc):
    return _accdm_common(f_acc, MASS_GRID[-1], ETA_COLD)


In [ ]:
# structural sanity: mapping + required keys present, exact quadrature enforced
assert abs(eta_of(1e11) - 1.0) < 1e-12
assert abs(eta_of(1e18) - 1e-7) < 1e-15
_p = accdm_params(0.1, 1e15)
assert _p['ncdm_quadrature_strategy'].endswith('4')      # daughter exact
assert _p['ncdm_fluid_approximation'] == 0               # never fluid
assert _p['m_ncdm'].split(',')[1].strip() == '{:.6e}'.format(1e15*1e9)
assert _p['eta_acc'] == eta_of(1e15)
assert coldlimit_params(0.1)['eta_acc'] == ETA_COLD
print('Task 1 param builders OK')


In [ ]:
_cache = {}
def run(params, key):
    """Compute a CLASS model once; return P(k) on K_NODES and lensed Cl (l=2..L_MAX)."""
    if key in _cache:
        return _cache[key]
    M = Class(); M.set(params); M.compute()
    pk = np.array([M.pk(float(k), 0.0) for k in K_NODES])
    cl = M.lensed_cl(L_MAX)
    ell = cl['ell']
    sel = ell >= 2
    out = {'pk': pk, 'ell': ell[sel],
           'tt': cl['tt'][sel], 'te': cl['te'][sel], 'ee': cl['ee'][sel]}
    M.struct_cleanup(); M.empty()
    _cache[key] = out
    return out

def get_lcdm():
    return run(lcdm_params(), ('lcdm',))

def get_cold(f_acc):
    return run(coldlimit_params(f_acc), ('cold', f_acc))

def get_warm(f_acc, m):
    return run(accdm_params(f_acc, m), ('warm', f_acc, m))


## Smoke test: one warm run + LCDM reference

In [ ]:
_warm = get_warm(0.1, 1e13)          # eta = 0.01
_lcdm = get_lcdm()
assert _warm['pk'].shape == K_NODES.shape
assert _warm['tt'].shape == _warm['ell'].shape
assert np.all(np.isfinite(_warm['pk'])) and np.all(np.isfinite(_warm['tt']))
assert _warm['ell'][0] == 2 and _warm['ell'][-1] == L_MAX
print('Task 2 run/cache OK: pk[0]={:.3e}, tt[100]={:.3e}'.format(_warm['pk'][0], _warm['tt'][100]))


## Metrics (pure functions) + unit tests

In [ ]:
# --- unit tests (defined before the functions on purpose; run after next cell) ---
def _test_metrics():
    k = np.logspace(-3, 0, 60)
    a = np.ones_like(k) * 2.0
    # identical spectra -> zero on every metric
    assert pk_maxdev(a, a) == 0.0
    assert pk_significance(a, a, k, 1e6) == 0.0
    assert cl_maxdev(a, a) == 0.0
    # max fractional deviation is exact
    b = a.copy(); b[10] = a[10] * 1.05
    assert abs(pk_maxdev(b, a) - 0.05) < 1e-12
    assert abs(cl_maxdev(b, a) - 0.05) < 1e-12
    # pk_significance matches the hand formula on a flat 1% offset
    ref = np.ones_like(k); acc = ref * 1.01
    dk = np.gradient(k); Nm = k**2 * dk * 1e6 / (2*np.pi**2)
    want = np.sqrt(np.sum((0.01 / np.sqrt(2.0/Nm))**2))
    assert abs(pk_significance(acc, ref, k, 1e6) - want) < 1e-9
    # cmb_significance is zero for identical run-dicts
    d = {'ell': np.arange(2, 50), 'tt': np.ones(48), 'te': np.zeros(48), 'ee': np.ones(48)}
    assert cmb_significance(d, d) == 0.0
    print('metric unit tests PASSED')

In [ ]:
def pk_maxdev(pk_acc, pk_ref):
    return float(np.max(np.abs(pk_acc / pk_ref - 1.0)))

def pk_significance(pk_acc, pk_ref, k=K_NODES, V=V_SURVEY):
    dk = np.gradient(k)
    N_modes = k**2 * dk * V / (2.0 * np.pi**2)
    sigmaP_over_P = np.sqrt(2.0 / N_modes)
    resid = (pk_acc - pk_ref) / pk_ref / sigmaP_over_P
    return float(np.sqrt(np.sum(resid**2)))

def cl_maxdev(cl_acc, cl_ref):
    return float(np.max(np.abs(cl_acc / cl_ref - 1.0)))

def cmb_significance(acc, ref, f_sky=F_SKY):
    """Cosmic-variance-limited Knox chi over TT+EE+TE (cross-covariance neglected)."""
    ell = ref['ell']; norm = (2.0 * ell + 1.0) * f_sky
    tt_r, ee_r, te_r = ref['tt'], ref['ee'], ref['te']
    var_tt = 2.0 * tt_r**2 / norm
    var_ee = 2.0 * ee_r**2 / norm
    var_te = (te_r**2 + tt_r * ee_r) / norm
    sig2 = np.sum((acc['tt'] - tt_r)**2 / var_tt)
    sig2 += np.sum((acc['ee'] - ee_r)**2 / var_ee)
    sig2 += np.sum((acc['te'] - te_r)**2 / var_te)
    return float(np.sqrt(sig2))

In [ ]:
_test_metrics()

In [ ]:
def _test_threshold():
    m = np.logspace(11, 19, 9)
    vals = np.array([1.0, 0.8, 0.6, 0.4, 0.2, 0.1, 0.05, 0.02, 0.01])   # decreasing
    # cut between vals[4]=0.2 and vals[5]=0.1 -> mass between m[4] and m[5]
    t = threshold_mass(m, vals, 0.15)
    assert m[4] < t < m[5]
    # already below everywhere -> smallest mass
    assert threshold_mass(m, vals*1e-3, 0.15) == m[0]
    # never below -> inf
    assert threshold_mass(m, vals*1e3, 0.15) == np.inf
    print('threshold unit tests PASSED')


In [ ]:
def threshold_mass(masses, metric_values, cut):
    masses = np.asarray(masses, float); vals = np.asarray(metric_values, float)
    logm = np.log10(masses)
    below = vals < cut
    if below.all():
        return float(masses[0])
    if not below.any():
        return np.inf
    i = int(np.argmax(below))            # first index below the cut
    if i == 0:
        return float(masses[0])
    x0, x1, y0, y1 = logm[i-1], logm[i], vals[i-1], vals[i]
    xc = x0 + (cut - y0) * (x1 - x0) / (y1 - y0)
    return float(10**xc)


In [ ]:
_test_threshold()

## Scan: distinguishability vs mass, per f_acc and per reference

In [ ]:
ETA_GRID = eta_of(MASS_GRID)
_lcdm = get_lcdm()
RESULTS = {}
for f_acc in F_SEQ:
    cold = get_cold(f_acc)
    per_ref = {r: {k: np.empty(len(MASS_GRID))
                   for k in ('pk_dev', 'pk_sig', 'cmb_dev', 'cmb_sig')}
               for r in ('cold', 'lcdm')}
    for j, m in enumerate(MASS_GRID):
        warm = get_warm(f_acc, m)
        for ref_name, ref in (('cold', cold), ('lcdm', _lcdm)):
            per_ref[ref_name]['pk_dev'][j]  = pk_maxdev(warm['pk'], ref['pk'])
            per_ref[ref_name]['pk_sig'][j]  = pk_significance(warm['pk'], ref['pk'])
            per_ref[ref_name]['cmb_dev'][j] = cl_maxdev(warm['tt'], ref['tt'])
            per_ref[ref_name]['cmb_sig'][j] = cmb_significance(warm, ref)
    RESULTS[f_acc] = per_ref
    print('f_acc={:<5} done  (pk_dev vs cold: {:.3e} -> {:.3e})'.format(
        f_acc, per_ref['cold']['pk_dev'][0], per_ref['cold']['pk_dev'][-1]))

In [ ]:
for f_acc in F_SEQ:
    for ref_name in ('cold', 'lcdm'):
        for k, arr in RESULTS[f_acc][ref_name].items():
            assert arr.shape == MASS_GRID.shape and np.all(np.isfinite(arr)), (f_acc, ref_name, k)
print('Task 5 RESULTS complete and finite')

## Threshold masses

In [ ]:
CUTS = {'pk_dev': 0.01, 'pk_sig': 1.0, 'cmb_dev': 0.01, 'cmb_sig': 1.0}  # indistinguishability cuts
THRESHOLDS = {}
print('{:>6} {:>6} {:>10} {:>10} {:>10} {:>10}'.format(
    'f_acc', 'ref', 'pk_dev', 'pk_sig', 'cmb_dev', 'cmb_sig'))
for f_acc in F_SEQ:
    THRESHOLDS[f_acc] = {}
    for ref_name in ('cold', 'lcdm'):
        row = {mk: threshold_mass(MASS_GRID, RESULTS[f_acc][ref_name][mk], CUTS[mk])
               for mk in CUTS}
        THRESHOLDS[f_acc][ref_name] = row
        fmt = lambda x: '{:.2e}'.format(x) if np.isfinite(x) else '  >grid'
        print('{:>6} {:>6} {:>10} {:>10} {:>10} {:>10}'.format(
            f_acc, ref_name, fmt(row['pk_dev']), fmt(row['pk_sig']),
            fmt(row['cmb_dev']), fmt(row['cmb_sig'])))
print('\nThreshold = daughter mass [GeV] above which accDM is indistinguishable from that reference.')
print('Note: "lcdm" rows are NOT background-matched (omega_cdm un-rescaled) -> informational only; '
      '"cold" rows are the quantitative reference.')

In [ ]:
metrics = [('pk_dev', r'$\max_k|P_{\rm acc}/P_{\rm ref}-1|$', CUTS['pk_dev'], True),
           ('pk_sig', r'$P(k)$ significance $\sqrt{\chi^2}$', CUTS['pk_sig'], True),
           ('cmb_dev', r'$\max_\ell|C^{TT}_{\rm acc}/C^{TT}_{\rm ref}-1|$', CUTS['cmb_dev'], True),
           ('cmb_sig', r'CMB significance $\sqrt{\chi^2}$', CUTS['cmb_sig'], True)]
fig, axes = plt.subplots(2, 2, figsize=(11, 8), constrained_layout=True)
for ax, (mk, ylab, cut, logy) in zip(axes.ravel(), metrics):
    for f_acc, c in zip(F_SEQ, qual_colors):
        ax.plot(MASS_GRID, RESULTS[f_acc]['cold'][mk], '-',  color=c,
                label=r'$f_{\rm acc}=%g$ (cold)' % f_acc)
        ax.plot(MASS_GRID, RESULTS[f_acc]['lcdm'][mk], '--', color=c, alpha=0.7)
    ax.axhline(cut, color='k', ls=':', lw=1.2)
    ax.set_xscale('log');  ax.set_xlabel(r'$m\ [\mathrm{GeV}]$');  ax.set_ylabel(ylab)
    if logy: ax.set_yscale('log')
    ax.grid(True, which='both', alpha=0.3)
axes[0, 0].legend(loc='best', fontsize=8)
fig.suptitle(r'accDM $\to$ CDM: solid = vs cold limit, dashed = vs $\Lambda$CDM (not bg-matched, informational)  '
             r'(dotted = indistinguishability cut)')
plt.show()

In [ ]:
f_show = 0.1
m_show = [MASS_GRID[0], MASS_GRID[len(MASS_GRID)//2], MASS_GRID[-1]]
cold = get_cold(f_show)
fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
for m, c in zip(m_show, qual_colors):
    w = get_warm(f_show, m)
    axL.semilogx(K_NODES, w['pk']/cold['pk'], color=c,
                 label=r'$m=%.0e$ GeV ($\eta=%.1e$)' % (m, eta_of(m)))
    axR.semilogx(cold['ell'], w['tt']/cold['tt'] - 1.0, color=c)
axL.axhline(1, color='k', ls='--', lw=1); axL.set_xlabel(r'$k\ [\mathrm{Mpc}^{-1}]$')
axL.set_ylabel(r'$P_{\rm acc}/P_{\rm cold}$'); axL.legend(fontsize=8); axL.grid(True, which='both', alpha=0.3)
axR.axhline(0, color='k', ls='--', lw=1); axR.set_xlabel(r'$\ell$')
axR.set_ylabel(r'$\Delta C_\ell^{TT}/C_\ell^{TT}$'); axR.grid(True, which='both', alpha=0.3)
fig.suptitle(r'Free-streaming signature shrinks as $m$ grows ($f_{\rm acc}=0.1$, vs cold limit)')
plt.show()

## q-grid convergence: is the cold-end floor physical or numerical?

In [ ]:
# Re-run the coldest mass at a finer daughter grid; the accDM<->accDM difference
# bounds the numeric floor. Thresholds sitting below Q_FLOOR are numerics-limited.
_m = MASS_GRID[-1]
p_coarse = accdm_params(0.1, _m)
p_fine = accdm_params(0.1, _m); p_fine['ncdm_N_momentum_bins'] = '15, {:d}'.format(4*Q_BINS)
coarse = run(p_coarse, ('warm', 0.1, _m))                 # cached from the scan
fine   = run(p_fine,   ('qconv', 0.1, _m, 4*Q_BINS))
Q_FLOOR = pk_maxdev(coarse['pk'], fine['pk'])
print('q-grid numeric floor on max|dP/P|  = {:.2e}  (Q_BINS={} vs {})'.format(
    Q_FLOOR, Q_BINS, 4*Q_BINS))
print('P(k) tolerance cut = {:.0e}. Floor is {} the cut.'.format(
    CUTS['pk_dev'], 'BELOW' if Q_FLOOR < CUTS['pk_dev'] else 'ABOVE'))
if Q_FLOOR >= CUTS['pk_dev']:
    print('WARNING: q-grid floor {:.2e} exceeds the P(k) cut {:.0e}. Cold-end thresholds are '
          'numerics-limited (not physical) at this Q_BINS; raise Q_BINS for a physical threshold.'
          .format(Q_FLOOR, CUTS['pk_dev']))

## Regression asserts (colder -> more CDM-like)

In [ ]:
def _mostly_decreasing(a, tol=0.15):
    """Allow small non-monotonic wiggles from q-grid noise; require net decrease."""
    a = np.asarray(a)
    steps = np.diff(a)
    assert a[0] > a[-1], 'metric must be smaller at the cold (large-mass) end'
    assert np.mean(steps <= tol*abs(a[0])) > 0.7, 'metric not broadly decreasing with mass'

for f_acc in F_SEQ:
    for ref_name in ('cold', 'lcdm'):
        _mostly_decreasing(RESULTS[f_acc][ref_name]['pk_dev'])
        _mostly_decreasing(RESULTS[f_acc][ref_name]['pk_sig'])
    # at the heaviest mass, accDM must be within the fixed P(k) tolerance of the cold limit
    floor = max(CUTS['pk_dev'], Q_FLOOR)
    assert RESULTS[f_acc]['cold']['pk_dev'][-1] < floor, \
        'cold-end P(k) distinguishable beyond the numeric floor at f_acc=%g (dev=%.2e, floor=%.2e)' \
        % (f_acc, RESULTS[f_acc]['cold']['pk_dev'][-1], floor)
print('Regression asserts PASSED (monotone trend + cold-end indistinguishability)')

### Notes and caveats
- **Two "CDM" references.** Solid curves compare to the model's own cold limit (`eta -> 0`),
  isolating free-streaming — this is the quantitative reference the asserts use. Dashed curves
  compare to plain LCDM, which is **not background-matched** (its `omega_cdm` is un-rescaled, so
  the total DM differs from the accDM runs by ~10%, cf. notebook 5's large-scale offset); the
  LCDM columns/curves are therefore **informational only**, not a quantitative indistinguishability
  statement.
- **Metrics.** (A) `max|ratio-1|` on the spectra vs a fixed tolerance; (B) cosmic-variance-limited
  `sqrt(chi^2)` (P(k): mode counting in `V_SURVEY`; CMB: Knox over TT+EE+TE, cross-covariance
  neglected as a detectability proxy — see the non-goals in the spec). These are optimistic
  ("if CV-limited can't tell them apart, nothing can").
- **q-grid floor.** The cold-end signal is tiny; the convergence cell bounds the numeric floor.
  Thresholds below `Q_FLOOR` are numerics-limited, not physical — raise `Q_BINS` to push lower.
- **CMB vs P(k).** For cold daughters the CMB effect enters mainly via late growth/lensing, so
  `P(k)` sets the binding (largest-mass) threshold; CMB typically becomes indistinguishable at
  lower mass.
- **Scope.** Fixed decay sector (`kappa_acc`, `a_t_acc`); no large-`kappa_acc` regime (known
  WONTFIX). The chi^2 is a single-parameter detectability proxy, not a Fisher/MCMC forecast.
- Run: `pytest --nbmake notebooks_test/13_test_accDM_CDM_indistinguishability.ipynb`.